# Proyecto 2 Bioseñales 

**Estudiantes:**

Luisa María Hernández Quintero 

Karen Agudelo Toro 

## Investigación de indices del EEG 

**El algoritmo de Patrones Espaciales Comunes (CSP):** es una técnica de filtrado espacial supervisada que está diseñada específicamente para problemas de clasificación binaria o multiclase en señales de EEG. A diferencia de los filtros temporales o espectrales (como Fourier o Welch) que observan cuándo y en qué frecuencia oscila la señal, CSP observa dónde ocurren los cambios de energía en la corteza cerebral [1].

Matemáticamente, el objetivo de CSP es transformar la matriz de datos original de los canales de EEG mediante una matriz de proyección lineal $W$. El criterio de optimización del algoritmo consiste en encontrar direcciones espaciales (filtros) que maximicen la varianza ( en este caso la potencia) de la señal filtrada para una condición clínica o tarea motora, mientras minimizan simultáneamente la varianza para la otra condición [1].


El índice CSP es considerado uno de los descriptores más dicientes y eficientes en Interfaces Cerebro-Computador (BCI) dado que: 

* El cráneo y las capas meníngeas actúan como conductores que dispersan los potenciales eléctricos. Esto causa que el canal $Cz$ capte actividad de $C3$ o $C4$, difuminando los patrones. CSP resuelve esto combinando los canales ponderadamente para aislar matemáticamente la fuente exacta de la corteza sensoriomotora.

* Los fenómenos de Desincronización Relacionada con Eventos (ERD) provocan pequeñas caídas de amplitud en el hemisferio opuesto. CSP amplifica  este contraste. Mientras que en una gráfica de potencia normal la diferencia entre imaginación izquierda y derecha puede ser de apenas fracciones diminutas de voltios, despues de aplicar CSP la separación entre clases se vulve evidente. 

* Permite comprimir información de múltiples electrodos en solo un par de componentes espaciales virtuales, reteniendo únicamente la información geométrica útil para la predicción y descartando el ruido muscular o ambiental de fondo.

Dado que la señal de EEG se encuentra previamente filtrada por software en una banda de frecuencias de interés (como el ritmo Mu o Beta), la varianza de la señal en el tiempo equivale directamente a la potencia o amplitud de dicha banda. El índice numérico que se extrae para entregarle al clasificador es el logaritmo de la varianza de estas señales ya proyectadas y limpias.

En este proyecto, el CSP opera sobre el Bloque 1 de datos (b1_ei y b1_ed), el cual conserva un mapa extendido de 7 canales del área motora (['C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6']). El algoritmo requiere obligatoriamente una estructura binaria etiquetada (Izquierda frente a Derecha) para entrenar sus matrices de covarianza. El proceso se ejecuta de forma independiente para  ['mu'] y para  ['beta']. Las señales de los 7 canales físicos se proyectan en componentes virtuales, se calcula su varianza logarítmica por cada época de 2 segundos y estos índices limpios se usan para alimentar clasificadores. 


**Coherencia Espectral de Magnitud Cuadrada (MSC)**

La Coherencia Espectral es una métrica estadística utilizada para cuantificar el grado de sincronización lineal o acoplamiento funcional entre dos señales en el dominio de la frecuencia. A diferencia de la Potencia Espectral (PSD), que evalúa la actividad de un solo electrodo de forma aislada, la coherencia examina la interacción y la conectividad de la red cerebral [2].Matemáticamente, la coherencia de magnitud cuadrada $C_{xy}(f)$ entre dos canales $x(t)$ y $y(t)$ a una frecuencia dada $f$ se define mediante la ecuación:$$C_{xy}(f) = \frac{|P_{xy}(f)|^2}{P_{xx}(f) P_{yy}(f)}$$Donde $P_{xy}(f)$ representa la densidad espectral cruzada entre ambas señales, mientras que $P_{xx}(f)$ y $P_{yy}(f)$ son las densidades espectrales de potencia de cada canal por separado [2]. El resultado de esta ecuación es un índice adimensional acotado estrictamente entre 0 y 1, donde 0 indica una independencia matemática absoluta entre los electrodos y 1 representa una sincronización perfecta de fase y amplitud.

Este índice es fundamental para un sistema BCI porque la imaginación motora no solo altera la energía local de un electrodo, sino que reorganiza las autopistas de comunicación interhemisférica. Durante el reposo, las áreas motoras izquierda y derecha suelen mantener una alta coherencia de fondo. Cuando el sujeto imagina un movimiento unilateral, la red se desincroniza de forma asimétrica, provocando una caída drástica de la coherencia entre el hemisferio que ejecuta la orden y el resto de la corteza, lo que proporciona un patrón altamente diciente para la clasificación.

La coherencia se calcula sobre el Bloque 0 (b0_er, b0_ei, b0_ed), el cual contiene los datos en crudo (filtrados de 0.5 a 45 Hz) de los tres canales principales: $C3$, $Cz$ y $C4$. Al ser una medida que puede evaluarse por condición independiente, se calcula la coherencia para cada época en tres pares de canales lógicos: $C3-C4$ (interhemisférica), $C3-Cz$ (izq-centro) y $C4-Cz$ (der-centro). La función devuelve una curva de coherencia sobre el espectro, y de forma idéntica a la PSD, se extrae el promedio del rango de $8-13\text{ Hz}$ para el índice en Mu, y de $13-30\text{ Hz}$ para el índice en Beta.



**Entropía de Permutación (PE)** 

La Entropía de Permutación es una medida de codificación temporal no lineal diseñada para cuantificar el grado de desorden, complejidad o regularidad dinámica de una serie de tiempo. Fue introducida por Bandt y Pompe y destaca por su capacidad de analizar sistemas biológicos complejos y caóticos como el cerebro, siendo sumamente robusta ante el ruido de fondo [3].El algoritmo opera convirtiendo la señal continua de EEG en una secuencia de patrones u ordenamientos ordinales (motivos espaciales) basados en la comparación de amplitudes de puntos de tiempo vecinos, definidos por un orden de incrustación $m$ y un retraso temporal $\tau$. Tras mapear la frecuencia de aparición de cada posible combinación u ordenamiento de ondas, se calcula la entropía de Shannon sobre dicha distribución de probabilidad [3].Una señal perfectamente predecible, repetitiva o periódica generará una entropía cercana a 0.Una señal caótica, desordenada y con alta variabilidad generará una entropía cercana a 1.

En el procesamiento de EEG, la entropía es un índice sumamente diciente para detectar los cambios sutiles del estado mental. Cuando el sujeto se encuentra en reposo, las neuronas de la corteza motora disparan de forma sincronizada, produciendo un ritmo rítmico, suave y predecible (baja entropía). Al iniciar la imaginación motora, las poblaciones neuronales se desincronizan (ERD) para procesar la información, volviendo la señal del EEG extremadamente compleja, impredecible y desordenada, lo que se traduce en un incremento medible en el valor de la entropía.

La entropía requiere evaluar canales de forma individual y sobre ondas de tiempo específicas de un ritmo. Para ello, se utiliza el Bloque 2 (b2_er, b2_ei, b2_ed), el cual entrega las épocas segmentadas por condición, reducidas a los canales $C3, Cz, C4$ y filtradas previamente en el tiempo por software. Para cada época de 2 segundos, se extrae el arreglo numérico del canal y se calcula secuencialmente el índice de entropía tanto para la señal sintonizada en Mu (['mu']) como para la señal en Beta (['beta']), generando variables estables y limpias en el dominio del tiempo para entrenar los clasificadore

In [1]:
import mne
import matplotlib.pyplot as plt
import numpy as np
from tabulate import tabulate
import pandas as pd
import os
import mne
import os
import glob
import numpy as np
import pandas as pd
from tabulate import tabulate

In [2]:
import os
import mne

def procesar_archivo_unificada(ruta):
    nombre = os.path.basename(ruta)
    partes = nombre.split('_')
    
    # Se extrae el sujeto y el run pegados respetando tu lógica original
    sujeto_puro = partes[0]
    run_con_extension = partes[2].split('-')[1]
    run_puro = run_con_extension.split('.')[0]
    sujeto_run = f"{sujeto_puro}{run_puro}"
    
    # Carga de datos única
    raw = mne.io.read_raw_eeglab(ruta, preload=True, verbose=False)
    
    # Se aplica el filtro Notch a 60Hz y el pasa-banda genérico de 0.5 a 45 Hz
    raw.notch_filter(60, verbose=False)
    raw.filter(0.5, 45, verbose=False)
    
    # Manejo de eventos usando tu mapeo exacto de PhysioNet
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    
    eventos_dict = {
        'reposo':    event_id.get('TASK1T0') or event_id.get('TASK2T0'),
        'izquierda': event_id.get('TASK1T1') or event_id.get('TASK2T1'),
        'derecha':   event_id.get('TASK1T2') or event_id.get('TASK2T2')
    }
    
    # Se limpian los eventos inexistentes
    eventos_dict = {k: v for k, v in eventos_dict.items() if v is not None}
    
    if not eventos_dict:
        print(f"Advertencia: No se encontraron eventos esperados en {nombre}")
        return None, None, None, None, None, None, None, None, None, sujeto_run, run_puro

    # Se define la lista de canales garantizados en la corteza motora para el CSP
    canales_motores_csp = ['C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6']
    
    tmin, tmax = 0.0, 2.0
    
  
    # GENERACIÓN DEL BLOQUE 0: Para PSD y Coherencia Espectral
  
    raw_bloque0 = raw.copy().pick(['C3', 'Cz', 'C4'])
    epocas_b0 = mne.Epochs(raw_bloque0, events, event_id=eventos_dict, tmin=tmin, tmax=tmax, baseline=None, preload=True, verbose=False)
    
    b0_er = epocas_b0['reposo'] if 'reposo' in epocas_b0.event_id else None
    b0_ei = epocas_b0['izquierda'] if 'izquierda' in epocas_b0.event_id else None
    b0_ed = epocas_b0['derecha'] if 'derecha' in epocas_b0.event_id else None
    
   
    # GENERACIÓN DEL BLOQUE 1: Para Patrones Espaciales Comunes (CSP)
    raw_bloque1 = raw.copy().pick(canales_motores_csp)
    epocas_b1 = mne.Epochs(raw_bloque1, events, event_id=eventos_dict, tmin=tmin, tmax=tmax, baseline=None, preload=True, verbose=False)
    
    ep_rep_csp = epocas_b1['reposo'] if 'reposo' in epocas_b1.event_id else None
    ep_izq_csp = epocas_b1['izquierda'] if 'izquierda' in epocas_b1.event_id else None
    ep_der_csp = epocas_b1['derecha'] if 'derecha' in epocas_b1.event_id else None
    
    b1_er = {'mu': ep_rep_csp.copy().filter(8.0, 13.0, verbose=False), 'beta': ep_rep_csp.copy().filter(13.0, 30.0, verbose=False)} if ep_rep_csp is not None else None
    b1_ei = {'mu': ep_izq_csp.copy().filter(8.0, 13.0, verbose=False), 'beta': ep_izq_csp.copy().filter(13.0, 30.0, verbose=False)} if ep_izq_csp is not None else None
    b1_ed = {'mu': ep_der_csp.copy().filter(8.0, 13.0, verbose=False), 'beta': ep_der_csp.copy().filter(13.0, 30.0, verbose=False)} if ep_der_csp is not None else None
    
   
    # GENERACIÓN DEL BLOQUE 2: Para Entropía de Permutación
    b2_er = {'mu': b0_er.copy().filter(8.0, 13.0, verbose=False), 'beta': b0_er.copy().filter(13.0, 30.0, verbose=False)} if b0_er is not None else None
    b2_ei = {'mu': b0_ei.copy().filter(8.0, 13.0, verbose=False), 'beta': b0_ei.copy().filter(13.0, 30.0, verbose=False)} if b0_ei is not None else None
    b2_ed = {'mu': b0_ed.copy().filter(8.0, 13.0, verbose=False), 'beta': b0_ed.copy().filter(13.0, 30.0, verbose=False)} if b0_ed is not None else None
    
    return b0_er, b0_ei, b0_ed, b1_er, b1_ei, b1_ed, b2_er, b2_ei, b2_ed, sujeto_run, run_puro

In [3]:
# 1. Ejecutar la función para extraer todo
b0_er, b0_ei, b0_ed, b1_er, b1_ei, b1_ed, b2_er, b2_ei, b2_ed, s, r = procesar_archivo_unificada('sub-001_task-motion_run-4_eeg.set')

# 2. Imprimir los resultados de forma clara y directa
print(f"SUJETO: {s} | RUN: {r}\n")

print("--- BLOQUE 0: Para Potencia (PSD) y Coherencia ---")
print(f"Reposo (3 canales: C3, Cz, C4)    : {b0_er}")
print(f"Izquierda (3 canales: C3, Cz, C4) : {b0_ei}")
print(f"Derecha (3 canales: C3, Cz, C4)   : {b0_ed}\n")

print("--- BLOQUE 1: Para CSP (7 canales para geometría) ---")
print(f"Izquierda en Ritmo Mu (8-13 Hz)   : {b1_ei['mu'] if b1_ei else None}")
print(f"Izquierda en Ritmo Beta (13-30 Hz): {b1_ei['beta'] if b1_ei else None}")
print(f"Derecha en Ritmo Mu (8-13 Hz)     : {b1_ed['mu'] if b1_ed else None}")
print(f"Derecha en Ritmo Beta (13-30 Hz)  : {b1_ed['beta'] if b1_ed else None}\n")

print("--- BLOQUE 2: Para Entropía (3 canales filtrados) ---")
print(f"Reposo en Ritmo Mu (8-13 Hz)      : {b2_er['mu'] if b2_er else None}")
print(f"Izquierda en Ritmo Beta (13-30 Hz): {b2_ei['beta'] if b2_ei else None}")
print(f"Derecha en Ritmo Beta (13-30 Hz)  : {b2_ed['beta'] if b2_ed else None}")

SUJETO: sub-0014 | RUN: 4

--- BLOQUE 0: Para Potencia (PSD) y Coherencia ---
Reposo (3 canales: C3, Cz, C4)    : <Epochs | 15 events (all good), 0 – 2 s (baseline off), ~144 KiB, data loaded,
 'reposo': 15>
Izquierda (3 canales: C3, Cz, C4) : <Epochs | 8 events (all good), 0 – 2 s (baseline off), ~91 KiB, data loaded,
 'izquierda': 8>
Derecha (3 canales: C3, Cz, C4)   : <Epochs | 7 events (all good), 0 – 2 s (baseline off), ~84 KiB, data loaded,
 'derecha': 7>

--- BLOQUE 1: Para CSP (7 canales para geometría) ---
Izquierda en Ritmo Mu (8-13 Hz)   : <Epochs | 8 events (all good), 0 – 2 s (baseline off), ~174 KiB, data loaded,
 'izquierda': 8>
Izquierda en Ritmo Beta (13-30 Hz): <Epochs | 8 events (all good), 0 – 2 s (baseline off), ~174 KiB, data loaded,
 'izquierda': 8>
Derecha en Ritmo Mu (8-13 Hz)     : <Epochs | 7 events (all good), 0 – 2 s (baseline off), ~157 KiB, data loaded,
 'derecha': 7>
Derecha en Ritmo Beta (13-30 Hz)  : <Epochs | 7 events (all good), 0 – 2 s (baseline off

In [4]:

def calcular_psd_epochs(epocas):
    """
    Calcula la Densidad Espectral de Potencia (PSD) usando el método de Welch
    para las épocas de los canales seleccionados (C3, Cz, C4).
    """
    # Verificación de seguridad por si el archivo fue descartado previamente
    if epocas is None:
        return None, None
        
    psds_obj = epocas.compute_psd(
        method='welch',
        fmin=0.5,       # Alineado con el filtro pasa-altas de tu función anterior
        fmax=45.0,      # Alineado con el filtro pasa-bajas de tu función anterior
        n_fft=256,
        n_overlap=128,
        verbose=False
    )
    
    # Extraemos los datos (epochs, canales, frecuencias) y el vector de frecuencias
    return psds_obj.get_data(), psds_obj.freqs


def potencia_banda(psds, freqs, fmin, fmax):
    """
    Calcula la potencia promedio en un rango de frecuencias específico (fmin a fmax)
    para cada época y cada canal.
    """
    if psds is None or freqs is None:
        return None
        
    # Crear una máscara booleana para seleccionar las frecuencias deseadas
    idx = (freqs >= fmin) & (freqs <= fmax)
    
    # psds tiene la forma: (n_epocas, n_canales, n_frecuencias)
    # Calculamos el promedio en el eje 2 (frecuencias)
    potencia = np.mean(psds[:, :, idx], axis=2)
    
    # Retorna una matriz de forma (n_epocas, n_canales) -> (Épocas, 3 canales)
    return potencia

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.decoding import CSP

def extraer_features_csp(b1_ei, b1_ed, ritmo='mu'):
    """
    Entrena el CSP con las épocas del Bloque 1 y retorna un DataFrame 
    con las varianzas logarítmicas de los componentes espaciales.
    """
    if b1_ei is None or b1_ed is None:
        print("Faltan épocas de izquierda o derecha para este archivo.")
        return None, None
    
    # 1. Extraer los datos numéricos de MNE [épocas, canales, tiempos]
    X_izq = b1_ei[ritmo].get_data()  
    X_der = b1_ed[ritmo].get_data()  
    
    # 2. Crear las etiquetas de clase (1 para izquierda, 2 para derecha)
    y_izq = np.ones(X_izq.shape[0]) * 1
    y_der = np.ones(X_der.shape[0]) * 2
    
    # Unificar en una sola masa de datos para el entrenamiento
    X_total = np.concatenate((X_izq, X_der), axis=0)
    y_total = np.concatenate((y_izq, y_der), axis=0)
    
    # 3. Inicializar el CSP para extraer los 4 componentes espaciales más fuertes
    # log=True calcula el logaritmo de la varianza automáticamente
    csp_modelo = CSP(n_components=4, reg=None, log=True, transform_into='average_power')
    
    # 4. Entrenar y transformar las señales continuas en índices
    indices_proyectados = csp_modelo.fit_transform(X_total, y_total)
    
    # 5. Estructurar en un DataFrame Ancho limpio
    columnas = [f'CSP_{ritmo}_comp1', f'CSP_{ritmo}_comp2', f'CSP_{ritmo}_comp3', f'CSP_{ritmo}_comp4']
    df_csp = pd.DataFrame(indices_proyectados, columns=columnas)
    
    # Añadir las columnas de control para el clasificador
    df_csp['tarea'] = y_total.astype(int)
    
    return df_csp, csp_modelo

In [6]:
# 2. Extraer las características CSP para el ritmo Mu
df_resultado_csp, modelo_entrenado = extraer_features_csp(b1_ei, b1_ed, ritmo='mu')

# 3. Imprimir el resultado de la matriz en la consola
print("MATRIZ DE CARACTERÍSTICAS (DATAFRAME ANCHO CSP):")
print(df_resultado_csp.head(6))

Computing rank from data with rank=None
    Using tolerance 3.8e-06 (2.2e-16 eps * 7 dim * 2.5e+09  max singular value)
    Estimated rank (data): 7
    data: rank 7 computed from 7 data channels with 0 projectors
Reducing data rank from 7 -> 7
Estimating class=1.0 covariance using EMPIRICAL
Done.
Estimating class=2.0 covariance using EMPIRICAL
Done.
MATRIZ DE CARACTERÍSTICAS (DATAFRAME ANCHO CSP):
   CSP_mu_comp1  CSP_mu_comp2  CSP_mu_comp3  CSP_mu_comp4  tarea
0     -1.044978     -0.749485     -0.857227     -0.994729      1
1     -1.326582     -0.813963     -0.760830     -0.872115      1
2     -0.961956     -0.402067     -0.986275     -1.238225      1
3     -1.694510     -0.519278      0.167844     -0.738951      1
4     -1.299064     -0.754659     -0.594468     -0.600929      1
5     -0.749108      0.011528     -0.652452     -0.701570      1


comp1 (El especialista en la Izquierda): Es un filtro diseñado para amplificar al máximo la señal cuando imaginas mover la mano izquierda y apagarla por completo cuando imaginas la derecha. 

comp2 (El asistente de la Izquierda): Hace un trabajo similar al primero, pero captura detalles secundarios o sutiles del hemisferio derecho del cerebro que ayudan a confirmar la tarea.

comp3 (El asistente de la Derecha): Es el gemelo del componente 2, pero empieza a buscar patrones que favorecen la detección de la mano derecha.

comp4 (El especialista en la Derecha): Es el filtro diseñado para hacer lo contrario al primero: amplifica la señal al máximo cuando imaginas mover la mano derecha y la destruye por completo si piensas en la izquierda.

In [8]:
import pandas as pd
import numpy as np

def construir_dataframe_ancho(ep_reposo, ep_izq, ep_der, sujeto_run):
    filas = []
    
    # Mapeo de cada objeto de épocas con su respectivo código numérico
    tareas_mapeo = {
        'reposo': (ep_reposo, 0),
        'izquierda': (ep_izq, 1),
        'derecha': (ep_der, 2)
    }
    
    for nombre_tarea, (objeto_epocas, codigo_tarea) in tareas_mapeo.items():
        if objeto_epocas is None:
            continue
            
        # Se obtiene la PSD de las épocas actuales
        psds, freqs = calcular_psd_epochs(objeto_epocas)
        
        # Se calcula la potencia cruda para las bandas Mu (8-13 Hz) y Beta (13-30 Hz)
        potencia_mu_cruda = potencia_banda(psds, freqs, 8, 13)
        potencia_beta_cruda = potencia_banda(psds, freqs, 13, 30)
        
        # Se extrae la cantidad de épocas disponibles para esta tarea
        n_epocas = potencia_mu_cruda.shape[0]
        
        # Se estructura la fila horizontal por cada época con los valores de potencia reales
        for i in range(n_epocas):
            fila = {
                'Sujeto': sujeto_run,
                'tarea': codigo_tarea,
                'potencia mu_C3': potencia_mu_cruda[i, 0],
                'potencia mu_Cz': potencia_mu_cruda[i, 1],
                ' potencia mu_C4': potencia_mu_cruda[i, 2],
                'potencia beta_C3': potencia_beta_cruda[i, 0],
                'potencia beta_Cz': potencia_beta_cruda[i, 1],
                'potencia beta_C4': potencia_beta_cruda[i, 2]
            }
            filas.append(fila)
            
    return pd.DataFrame(filas)

In [ ]:
# Se ejecuta el procesamiento unificado del archivo original
ruta = 'sub-001_task-motion_run-4_eeg.set'
b0_er, b0_ei, b0_ed, b1_er, b1_ei, b1_ed, b2_er, b2_ei, b2_ed, sujeto_run, run = procesar_archivo_unificada(ruta)

# Se construye el DataFrame ancho pasando exclusivamente las épocas del Bloque 0
# tal como solicitaste, sin incluir los otros índices por el momento
df_potencia_real = construir_dataframe_ancho(b0_er, b0_ei, b0_ed, sujeto_run)

# Visualización del DataFrame resultante para verificar las columnas de potencia estructuradas
print(df_potencia_real.head())

     Sujeto  tarea  potencia mu_C3  potencia mu_Cz   potencia mu_C4  \
0  sub-0014      0    2.783223e-11    3.967592e-11     3.632389e-11   
1  sub-0014      0    2.206237e-11    2.278148e-11     1.280217e-11   
2  sub-0014      0    2.104106e-11    2.744501e-11     1.161609e-11   
3  sub-0014      0    2.790867e-11    1.626432e-11     1.992437e-11   
4  sub-0014      0    1.396609e-11    1.041238e-11     7.148334e-12   

   potencia beta_C3  potencia beta_Cz  potencia beta_C4  
0      8.323840e-12      1.154432e-11      1.090933e-11  
1      1.230438e-11      1.012310e-11      6.127661e-12  
2      8.225814e-12      6.925411e-12      5.324543e-12  
3      1.331262e-11      9.536927e-12      8.048151e-12  
4      9.168110e-12      1.080732e-11      9.498108e-12  


## Referencias 
[1] Blankertz, B., Tomioka, R., Lemm, S., Kawanabe, M., & Müller, K. R. (2007). Optimizing spatial filters for EEG-based brain–computer interfaces: a tutorial overview. IEEE Signal Processing Magazine, 24(1), 41-44. 

[2] Andrew, C., & Pfurtscheller, G. (1996). Event-related coherence as a tool for studying dynamic interaction of brain regions. Electroencephalography and Clinical Neurophysiology, 98(2), 144-148. 

[3] Bandt, C., & Pompe, B. (2002). Permutation entropy: a natural complexity measure for time series. Physical Review Letters, 88(17), 174102.